# 26 — Semisupervised Labeled / Unlabeled Split

## Objetivo

Materializar el primer conjunto `L ∪ U` a partir de los resultados ejecutados del notebook 25.

El notebook 25 demostró que:

- `STRICT_CANONICAL` está vacío;
- existe un espacio `EXPERIMENTAL_CONSTRUCT_PROXY` con tres variables;
- Catching aporta dos sitios con Y Nielsen directa;
- UICrit/RICO aporta 1000 interfaces sin Y Nielsen directa;
- Catching solo contiene clase positiva en H1/H4/H5/H6.

Por tanto, este notebook **no crea todavía train/test** y **no entrena modelos**.

Su función es:

1. construir un dataset maestro experimental con las tres X proxy;
2. mantener las Y Nielsen observadas únicamente en Catching;
3. convertir UICrit/RICO en `U` para Nielsen sin utilizar sus ratings como Y;
4. construir la pertenencia `L/U` separadamente para H1, H4, H5 y H6;
5. excluir filas con X incompleta sin imputarlas todavía;
6. registrar explícitamente el cambio de dominio `WEB_DOM ↔ ANDROID_UI`;
7. dejar los datos listos para la auditoría de entrenabilidad del notebook 27.

Este dataset es **experimental** y no debe describirse como un espacio canónico Nielsen.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"
PUBLIC_INTERIM_DIR = PROJECT_DIR / "data" / "public_v2" / "interim"

SEMISUP_DIR = PROJECT_DIR / "data" / "semisupervised"
SEMISUP_REPORTS_DIR = REPORTS_DIR / "semisupervised"
SEMISUP_INTERIM_DIR = SEMISUP_DIR / "interim"
TARGET_DIR = SEMISUP_INTERIM_DIR / "targets"

for directory in [
    SEMISUP_REPORTS_DIR,
    SEMISUP_INTERIM_DIR,
    TARGET_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

FEATURES = [
    "structure_depth",
    "interactive_control_count",
    "interactive_label_ratio"
]

print("PROJECT_DIR:", PROJECT_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning


## 2. Cargar decisiones y fuentes

In [2]:
feature_decision = pd.read_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_feature_space_decision.csv"
)

feature_space = pd.read_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_feature_space.csv"
)

catching = pd.read_csv(
    PUBLIC_INTERIM_DIR
    / "catching_static_validation_dataset.csv"
)

rico = pd.read_csv(
    PUBLIC_INTERIM_DIR
    / "uicrit_rico_proxy_dataset.csv"
)

vt_targets_path = (
    PUBLIC_INTERIM_DIR
    / "vt_recovered_candidate_targets.csv"
)

vt_targets = (
    pd.read_csv(vt_targets_path)
    if vt_targets_path.exists()
    else None
)

print("Feature decision:", feature_decision.shape)
print("Feature space:", feature_space.shape)
print("Catching:", catching.shape)
print("UICrit/RICO:", rico.shape)
print(
    "29042675 labels:",
    None if vt_targets is None else vt_targets.shape
)


Feature decision: (4, 3)
Feature space: (3, 5)
Catching: (2, 23)
UICrit/RICO: (1000, 23)
29042675 labels: (15, 5)


## 3. Verificar el escenario heredado del notebook 25

In [3]:
decision_map = dict(
    zip(
        feature_decision["decision"],
        feature_decision["value"]
    )
)

def as_bool(value):
    if isinstance(value, bool):
        return value

    return str(value).strip().lower() == "true"


strict_available = as_bool(
    decision_map.get(
        "STRICT_COMMON_X_AVAILABLE",
        False
    )
)

proxy_available = as_bool(
    decision_map.get(
        "EXPERIMENTAL_PROXY_X_AVAILABLE",
        False
    )
)

print("STRICT_COMMON_X_AVAILABLE:", strict_available)
print("EXPERIMENTAL_PROXY_X_AVAILABLE:", proxy_available)

assert strict_available is False
assert proxy_available is True

proxy_features_from_file = (
    feature_space.loc[
        feature_space["feature_space"]
        ==
        "EXPERIMENTAL_CONSTRUCT_PROXY",
        "model_feature"
    ]
    .astype(str)
    .tolist()
)

print(
    "Proxy features:",
    proxy_features_from_file
)

assert set(proxy_features_from_file) == set(FEATURES)


STRICT_COMMON_X_AVAILABLE: False
EXPERIMENTAL_PROXY_X_AVAILABLE: True
Proxy features: ['structure_depth', 'interactive_control_count', 'interactive_label_ratio']


## 4. Reglas de esta materialización

- Catching será la fuente `L` observada.
- UICrit/RICO será la fuente `U` para Nielsen.
- Los ratings de UICrit/RICO **no se copian** a H1/H4/H5/H6.
- `29042675` no entra a `L ∪ U` porque actualmente tiene Y pero no X compatible.
- Los dos valores faltantes de `rico_interactive_label_proxy_ratio` se conservan como `NaN`.
- Ningún `NaN` se convierte a cero.


## 5. Funciones de identidad

In [4]:
def normalize_id_part(value):
    value = str(value).strip().lower()
    value = re.sub(
        r"[^a-z0-9]+",
        "_",
        value
    )
    return value.strip("_")


def master_id(source_dataset, source_instance_id):
    return (
        normalize_id_part(source_dataset)
        + "__"
        + normalize_id_part(source_instance_id)
    )


## 6. Adaptar Catching al espacio proxy

In [5]:
required_catching_columns = [
    "site",
    "static_max_dom_depth",
    "static_total_interactive_controls",
    "static_control_label_proxy_ratio",
    "label_source",
    "label_level"
] + TARGETS

missing_catching = sorted(
    set(required_catching_columns)
    -
    set(catching.columns)
)

assert missing_catching == [], missing_catching

catching_master = pd.DataFrame({
    "master_instance_id": [
        master_id(
            "Catching_UX_Flaws",
            value
        )
        for value in catching["site"]
    ],

    "source_dataset":
        "Catching_UX_Flaws",

    "source_instance_id":
        catching["site"].astype(str),

    "source_unit":
        "website",

    "feature_domain":
        "WEB_DOM",

    "feature_space":
        "EXPERIMENTAL_CONSTRUCT_PROXY",

    "structure_depth":
        pd.to_numeric(
            catching[
                "static_max_dom_depth"
            ],
            errors="coerce"
        ),

    "interactive_control_count":
        pd.to_numeric(
            catching[
                "static_total_interactive_controls"
            ],
            errors="coerce"
        ),

    "interactive_label_ratio":
        pd.to_numeric(
            catching[
                "static_control_label_proxy_ratio"
            ],
            errors="coerce"
        ),

    "H1_violation":
        pd.to_numeric(
            catching["H1_violation"],
            errors="coerce"
        ),

    "H4_violation":
        pd.to_numeric(
            catching["H4_violation"],
            errors="coerce"
        ),

    "H5_violation":
        pd.to_numeric(
            catching["H5_violation"],
            errors="coerce"
        ),

    "H6_violation":
        pd.to_numeric(
            catching["H6_violation"],
            errors="coerce"
        ),

    "label_source":
        catching["label_source"].astype(str),

    "source_label_level":
        catching["label_level"].astype(str),

    "nielsen_label_level":
        "LLM_DIRECT",

    "is_pseudo_label":
        False,

    "group_id": [
        "catching__"
        + normalize_id_part(value)
        for value in catching["site"]
    ],

    "domain_shift_flag":
        "WEB_DOM"
})

catching_master[
    "complete_proxy_x"
] = catching_master[
    FEATURES
].notna().all(axis=1)

display(catching_master)


,master_instance_id,source_dataset,source_instance_id,source_unit,feature_domain,feature_space,structure_depth,interactive_control_count,interactive_label_ratio,H1_violation,H4_violation,H5_violation,H6_violation,label_source,source_label_level,nielsen_label_level,is_pseudo_label,group_id,domain_shift_flag,complete_proxy_x
0,catching_ux_flaws__startbootstrap_3_col_portfolio,Catching_UX_Flaws,startbootstrap-3-col-portfolio,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,8.0,23.0,0.739130,1,1.0,1,1,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__startbootstrap_3_col_portfolio,WEB_DOM,True
1,catching_ux_flaws__startbootstrap_sb_admin_2,Catching_UX_Flaws,startbootstrap-sb-admin-2,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,15.0,585.0,0.842735,1,1.0,1,1,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__startbootstrap_sb_admin_2,WEB_DOM,True


## 7. Validar Y observada de Catching

In [6]:
catching_label_audit = []

for target in TARGETS:
    values = pd.to_numeric(
        catching_master[target],
        errors="coerce"
    )

    invalid = (
        values.notna()
        &
        ~values.isin([0, 1])
    )

    catching_label_audit.append({
        "target": target,
        "n_rows": len(values),
        "n_labeled": int(values.notna().sum()),
        "n_0": int((values == 0).sum()),
        "n_1": int((values == 1).sum()),
        "n_invalid": int(invalid.sum())
    })

catching_label_audit = pd.DataFrame(
    catching_label_audit
)

display(catching_label_audit)

assert (
    catching_label_audit["n_invalid"]
    ==
    0
).all()


,target,n_rows,n_labeled,n_0,n_1,n_invalid
0,H1_violation,2,2,0,2,0
1,H4_violation,2,2,0,2,0
2,H5_violation,2,2,0,2,0
3,H6_violation,2,2,0,2,0


## 8. Adaptar UICrit/RICO al espacio proxy

In [7]:
required_rico_columns = [
    "rico_id",
    "rico_max_tree_depth",
    "rico_interactive_node_count",
    "rico_interactive_label_proxy_ratio",
    "label_level"
]

missing_rico = sorted(
    set(required_rico_columns)
    -
    set(rico.columns)
)

assert missing_rico == [], missing_rico

rico_master = pd.DataFrame({
    "master_instance_id": [
        master_id(
            "UICrit_RICO",
            value
        )
        for value in rico["rico_id"]
    ],

    "source_dataset":
        "UICrit_RICO",

    "source_instance_id":
        rico["rico_id"].astype(str),

    "source_unit":
        "android_ui",

    "feature_domain":
        "ANDROID_UI",

    "feature_space":
        "EXPERIMENTAL_CONSTRUCT_PROXY",

    "structure_depth":
        pd.to_numeric(
            rico[
                "rico_max_tree_depth"
            ],
            errors="coerce"
        ),

    "interactive_control_count":
        pd.to_numeric(
            rico[
                "rico_interactive_node_count"
            ],
            errors="coerce"
        ),

    "interactive_label_ratio":
        pd.to_numeric(
            rico[
                "rico_interactive_label_proxy_ratio"
            ],
            errors="coerce"
        ),

    "H1_violation":
        np.nan,

    "H4_violation":
        np.nan,

    "H5_violation":
        np.nan,

    "H6_violation":
        np.nan,

    "label_source":
        "NONE_FOR_NIELSEN",

    "source_label_level":
        rico["label_level"].astype(str),

    "nielsen_label_level":
        "NONE",

    "is_pseudo_label":
        False,

    "group_id": [
        "rico__"
        + normalize_id_part(value)
        for value in rico["rico_id"]
    ],

    "domain_shift_flag":
        "ANDROID_UI"
})

rico_master[
    "complete_proxy_x"
] = rico_master[
    FEATURES
].notna().all(axis=1)

display(
    rico_master.head()
)


,master_instance_id,source_dataset,source_instance_id,source_unit,feature_domain,feature_space,structure_depth,interactive_control_count,interactive_label_ratio,H1_violation,H4_violation,H5_violation,H6_violation,label_source,source_label_level,nielsen_label_level,is_pseudo_label,group_id,domain_shift_flag,complete_proxy_x
0,uicrit_rico__36217,UICrit_RICO,36217,android_ui,ANDROID_UI,EXPERIMENTAL_CONSTRUCT_PROXY,7,2,0.000000,NaN,NaN,NaN,NaN,NONE_FOR_NIELSEN,RATING_PROXY,NONE,False,rico__36217,ANDROID_UI,True
1,uicrit_rico__63093,UICrit_RICO,63093,android_ui,ANDROID_UI,EXPERIMENTAL_CONSTRUCT_PROXY,15,7,0.428571,NaN,NaN,NaN,NaN,NONE_FOR_NIELSEN,RATING_PROXY,NONE,False,rico__63093,ANDROID_UI,True
2,uicrit_rico__21067,UICrit_RICO,21067,android_ui,ANDROID_UI,EXPERIMENTAL_CONSTRUCT_PROXY,21,36,0.111111,NaN,NaN,NaN,NaN,NONE_FOR_NIELSEN,RATING_PROXY,NONE,False,rico__21067,ANDROID_UI,True
3,uicrit_rico__13332,UICrit_RICO,13332,android_ui,ANDROID_UI,EXPERIMENTAL_CONSTRUCT_PROXY,11,10,0.600000,NaN,NaN,NaN,NaN,NONE_FOR_NIELSEN,RATING_PROXY,NONE,False,rico__13332,ANDROID_UI,True
4,uicrit_rico__70793,UICrit_RICO,70793,android_ui,ANDROID_UI,EXPERIMENTAL_CONSTRUCT_PROXY,9,2,0.500000,NaN,NaN,NaN,NaN,NONE_FOR_NIELSEN,RATING_PROXY,NONE,False,rico__70793,ANDROID_UI,True


## 9. Comprobar que los ratings proxy no se filtraron a Y Nielsen

UICrit/RICO sí tiene ratings/variables proxy en su fuente original, pero aquí H1/H4/H5/H6 deben ser `NaN` en las 1000 filas.


In [8]:
rico_y_nonmissing = int(
    rico_master[
        TARGETS
    ]
    .notna()
    .sum()
    .sum()
)

print(
    "Celdas Nielsen no-NaN en RICO:",
    rico_y_nonmissing
)

assert rico_y_nonmissing == 0


Celdas Nielsen no-NaN en RICO: 0


## 10. Construir master experimental

In [9]:
master_proxy = pd.concat(
    [
        catching_master,
        rico_master
    ],
    axis=0,
    ignore_index=True
)

master_proxy[
    "has_any_observed_nielsen_y"
] = master_proxy[
    TARGETS
].notna().any(axis=1)

master_proxy[
    "master_role"
] = np.where(
    master_proxy[
        "has_any_observed_nielsen_y"
    ],
    "OBSERVED_LABEL_SOURCE",
    "UNLABELED_SOURCE_FOR_NIELSEN"
)

print(
    "Master shape:",
    master_proxy.shape
)

print(
    master_proxy[
        "source_dataset"
    ]
    .value_counts()
)

print(
    "\nComplete proxy X:"
)

print(
    master_proxy[
        "complete_proxy_x"
    ]
    .value_counts(
        dropna=False
    )
)

display(
    master_proxy.head()
)


Master shape: (1002, 22)
source_dataset
UICrit_RICO          1000
Catching_UX_Flaws       2
Name: count, dtype: int64

Complete proxy X:
complete_proxy_x
True     1000
False       2
Name: count, dtype: int64


,master_instance_id,source_dataset,source_instance_id,source_unit,feature_domain,feature_space,structure_depth,interactive_control_count,interactive_label_ratio,H1_violation,H4_violation,H5_violation,H6_violation,label_source,source_label_level,nielsen_label_level,is_pseudo_label,group_id,domain_shift_flag,complete_proxy_x,has_any_observed_nielsen_y,master_role
0,catching_ux_flaws__startbootstrap_3_col_portfolio,Catching_UX_Flaws,startbootstrap-3-col-portfolio,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,8.0,23.0,0.739130,1.0,1.0,1.0,1.0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__startbootstrap_3_col_portfolio,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE
1,catching_ux_flaws__startbootstrap_sb_admin_2,Catching_UX_Flaws,startbootstrap-sb-admin-2,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,15.0,585.0,0.842735,1.0,1.0,1.0,1.0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__startbootstrap_sb_admin_2,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE
2,uicrit_rico__36217,UICrit_RICO,36217,android_ui,ANDROID_UI,EXPERIMENTAL_CONSTRUCT_PROXY,7.0,2.0,0.000000,NaN,NaN,NaN,NaN,NONE_FOR_NIELSEN,RATING_PROXY,NONE,False,rico__36217,ANDROID_UI,True,False,UNLABELED_SOURCE_FOR_NIELSEN
3,uicrit_rico__63093,UICrit_RICO,63093,android_ui,ANDROID_UI,EXPERIMENTAL_CONSTRUCT_PROXY,15.0,7.0,0.428571,NaN,NaN,NaN,NaN,NONE_FOR_NIELSEN,RATING_PROXY,NONE,False,rico__63093,ANDROID_UI,True,False,UNLABELED_SOURCE_FOR_NIELSEN
4,uicrit_rico__21067,UICrit_RICO,21067,android_ui,ANDROID_UI,EXPERIMENTAL_CONSTRUCT_PROXY,21.0,36.0,0.111111,NaN,NaN,NaN,NaN,NONE_FOR_NIELSEN,RATING_PROXY,NONE,False,rico__21067,ANDROID_UI,True,False,UNLABELED_SOURCE_FOR_NIELSEN


## 11. Integridad de IDs y features

In [10]:
integrity_records = []

integrity_records.append({
    "check": "MASTER_ROWS",
    "observed": len(master_proxy),
    "expected": len(catching) + len(rico),
    "passed": (
        len(master_proxy)
        ==
        len(catching) + len(rico)
    )
})

duplicate_ids = int(
    master_proxy[
        "master_instance_id"
    ]
    .duplicated()
    .sum()
)

integrity_records.append({
    "check": "DUPLICATE_MASTER_ID",
    "observed": duplicate_ids,
    "expected": 0,
    "passed": duplicate_ids == 0
})

catching_complete = int(
    catching_master[
        "complete_proxy_x"
    ].sum()
)

integrity_records.append({
    "check": "CATCHING_COMPLETE_X",
    "observed": catching_complete,
    "expected": len(catching_master),
    "passed": catching_complete == len(catching_master)
})

rico_complete = int(
    rico_master[
        "complete_proxy_x"
    ].sum()
)

integrity_records.append({
    "check": "RICO_COMPLETE_X",
    "observed": rico_complete,
    "expected": "documented_not_forced",
    "passed": True
})

integrity_audit = pd.DataFrame(
    integrity_records
)

display(integrity_audit)


,check,observed,expected,passed
0,MASTER_ROWS,1002,1002,True
1,DUPLICATE_MASTER_ID,0,0,True
2,CATCHING_COMPLETE_X,2,2,True
3,RICO_COMPLETE_X,998,documented_not_forced,True


## 12. Construir pertenencia `L/U` por target

Definición:

- `L`: Y observada 0/1 + X proxy completa.
- `U`: Y desconocida + X proxy completa.
- `EXCLUDED_INCOMPLETE_X`: al menos una de las tres X falta.
- `EXCLUDED_INVALID_Y`: Y observada fuera de `{0,1}`.

Esto todavía **no** es un split train/test.


In [11]:
membership_records = []

for _, row in master_proxy.iterrows():

    for target in TARGETS:

        y = row[target]

        if not bool(
            row[
                "complete_proxy_x"
            ]
        ):
            role = "EXCLUDED_INCOMPLETE_X"
            reason = (
                "At least one proxy X is NaN."
            )

        elif pd.isna(y):
            role = "U"
            reason = (
                "No observed Nielsen label "
                "for this target."
            )

        elif y in [0, 1]:
            role = "L"
            reason = (
                "Observed binary Nielsen label "
                "and complete proxy X."
            )

        else:
            role = "EXCLUDED_INVALID_Y"
            reason = (
                "Observed target is not binary."
            )

        membership_records.append({
            "master_instance_id":
                row[
                    "master_instance_id"
                ],

            "source_dataset":
                row[
                    "source_dataset"
                ],

            "source_instance_id":
                row[
                    "source_instance_id"
                ],

            "group_id":
                row[
                    "group_id"
                ],

            "feature_domain":
                row[
                    "feature_domain"
                ],

            "target":
                target,

            "observed_y":
                y,

            "ssl_role":
                role,

            "reason":
                reason
        })

target_membership = pd.DataFrame(
    membership_records
)

display(
    target_membership.head(20)
)


,master_instance_id,source_dataset,source_instance_id,group_id,feature_domain,target,observed_y,ssl_role,reason
0,catching_ux_flaws__startbootstrap_3_col_portfolio,Catching_UX_Flaws,startbootstrap-3-col-portfolio,catching__startbootstrap_3_col_portfolio,WEB_DOM,H1_violation,1.0,L,Observed binary Nielsen label and complete proxy X.
1,catching_ux_flaws__startbootstrap_3_col_portfolio,Catching_UX_Flaws,startbootstrap-3-col-portfolio,catching__startbootstrap_3_col_portfolio,WEB_DOM,H4_violation,1.0,L,Observed binary Nielsen label and complete proxy X.
2,catching_ux_flaws__startbootstrap_3_col_portfolio,Catching_UX_Flaws,startbootstrap-3-col-portfolio,catching__startbootstrap_3_col_portfolio,WEB_DOM,H5_violation,1.0,L,Observed binary Nielsen label and complete proxy X.
3,catching_ux_flaws__startbootstrap_3_col_portfolio,Catching_UX_Flaws,startbootstrap-3-col-portfolio,catching__startbootstrap_3_col_portfolio,WEB_DOM,H6_violation,1.0,L,Observed binary Nielsen label and complete proxy X.
4,catching_ux_flaws__startbootstrap_sb_admin_2,Catching_UX_Flaws,startbootstrap-sb-admin-2,catching__startbootstrap_sb_admin_2,WEB_DOM,H1_violation,1.0,L,Observed binary Nielsen label and complete proxy X.
5,catching_ux_flaws__startbootstrap_sb_admin_2,Catching_UX_Flaws,startbootstrap-sb-admin-2,catching__startbootstrap_sb_admin_2,WEB_DOM,H4_violation,1.0,L,Observed binary Nielsen label and complete proxy X.
6,catching_ux_flaws__startbootstrap_sb_admin_2,Catching_UX_Flaws,startbootstrap-sb-admin-2,catching__startbootstrap_sb_admin_2,WEB_DOM,H5_violation,1.0,L,Observed binary Nielsen label and complete proxy X.
7,catching_ux_flaws__startbootstrap_sb_admin_2,Catching_UX_Flaws,startbootstrap-sb-admin-2,catching__startbootstrap_sb_admin_2,WEB_DOM,H6_violation,1.0,L,Observed binary Nielsen label and complete proxy X.
8,uicrit_rico__36217,UICrit_RICO,36217,rico__36217,ANDROID_UI,H1_violation,NaN,U,No observed Nielsen label for this target.
9,uicrit_rico__36217,UICrit_RICO,36217,rico__36217,ANDROID_UI,H4_violation,NaN,U,No observed Nielsen label for this target.


## 13. Resumen L/U por heurística

In [12]:
split_summary = (
    target_membership
    .groupby(
        [
            "target",
            "ssl_role"
        ],
        dropna=False
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

for col in [
    "L",
    "U",
    "EXCLUDED_INCOMPLETE_X",
    "EXCLUDED_INVALID_Y"
]:
    if col not in split_summary.columns:
        split_summary[col] = 0

split_summary[
    "usable_rows"
] = (
    split_summary["L"]
    +
    split_summary["U"]
)

display(
    split_summary[
        [
            "target",
            "L",
            "U",
            "EXCLUDED_INCOMPLETE_X",
            "EXCLUDED_INVALID_Y",
            "usable_rows"
        ]
    ]
)


ssl_role,target,L,U,EXCLUDED_INCOMPLETE_X,EXCLUDED_INVALID_Y,usable_rows
0,H1_violation,2,998,2,0,1000
1,H4_violation,2,998,2,0,1000
2,H5_violation,2,998,2,0,1000
3,H6_violation,2,998,2,0,1000


## 14. Balance de clase dentro de L

In [13]:
labeled_balance_records = []

for target in TARGETS:

    ids = target_membership.loc[
        (
            target_membership["target"]
            ==
            target
        )
        &
        (
            target_membership["ssl_role"]
            ==
            "L"
        ),
        "master_instance_id"
    ]

    values = (
        master_proxy
        .set_index(
            "master_instance_id"
        )
        .loc[
            ids,
            target
        ]
    )

    labeled_balance_records.append({
        "target": target,
        "n_labeled": int(values.notna().sum()),
        "n_0": int((values == 0).sum()),
        "n_1": int((values == 1).sum()),
        "n_classes": int(values.dropna().nunique())
    })

labeled_balance = pd.DataFrame(
    labeled_balance_records
)

display(labeled_balance)


,target,n_labeled,n_0,n_1,n_classes
0,H1_violation,2,0,2,1
1,H4_violation,2,0,2,1
2,H5_violation,2,0,2,1
3,H6_violation,2,0,2,1


## 15. Materializar L y U por target

Los archivos contienen únicamente las tres X proxy, metadata mínima y la Y correspondiente.

No contienen ratings de RICO ni variables fuente-específicas adicionales.


In [14]:
base_columns = [
    "master_instance_id",
    "source_dataset",
    "source_instance_id",
    "source_unit",
    "feature_domain",
    "group_id",
    "feature_space"
] + FEATURES

target_file_registry = []

for target in TARGETS:

    short = target.split("_")[0]

    role_table = target_membership[
        target_membership["target"]
        ==
        target
    ]

    labeled_ids = role_table.loc[
        role_table["ssl_role"] == "L",
        "master_instance_id"
    ].tolist()

    unlabeled_ids = role_table.loc[
        role_table["ssl_role"] == "U",
        "master_instance_id"
    ].tolist()

    L_target = master_proxy[
        master_proxy[
            "master_instance_id"
        ].isin(labeled_ids)
    ][
        base_columns
        +
        [
            target,
            "label_source",
            "nielsen_label_level",
            "is_pseudo_label"
        ]
    ].copy()

    U_target = master_proxy[
        master_proxy[
            "master_instance_id"
        ].isin(unlabeled_ids)
    ][
        base_columns
        +
        [
            target,
            "label_source",
            "nielsen_label_level",
            "is_pseudo_label"
        ]
    ].copy()

    l_path = (
        TARGET_DIR
        /
        f"{short}_labeled_seed.csv"
    )

    u_path = (
        TARGET_DIR
        /
        f"{short}_unlabeled_pool.csv"
    )

    L_target.to_csv(
        l_path,
        index=False
    )

    U_target.to_csv(
        u_path,
        index=False
    )

    target_file_registry.append({
        "target": target,
        "labeled_file": str(
            l_path.relative_to(
                PROJECT_DIR
            )
        ),
        "n_L": len(L_target),
        "unlabeled_file": str(
            u_path.relative_to(
                PROJECT_DIR
            )
        ),
        "n_U": len(U_target)
    })

target_file_registry = pd.DataFrame(
    target_file_registry
)

display(target_file_registry)


,target,labeled_file,n_L,unlabeled_file,n_U
0,H1_violation,data/semisupervised/interim/targets/H1_labeled_seed.csv,2,data/semisupervised/interim/targets/H1_unlabeled_pool.csv,998
1,H4_violation,data/semisupervised/interim/targets/H4_labeled_seed.csv,2,data/semisupervised/interim/targets/H4_unlabeled_pool.csv,998
2,H5_violation,data/semisupervised/interim/targets/H5_labeled_seed.csv,2,data/semisupervised/interim/targets/H5_unlabeled_pool.csv,998
3,H6_violation,data/semisupervised/interim/targets/H6_labeled_seed.csv,2,data/semisupervised/interim/targets/H6_unlabeled_pool.csv,998


## 16. Registrar fuentes excluidas de esta materialización

Excluir una fuente del `L ∪ U` actual no significa descartar el dataset del proyecto.

Solo significa que no puede participar en **este espacio de tres X proxy** con la evidencia disponible.


In [15]:
excluded_sources = pd.DataFrame([
    {
        "source_dataset":
            "29042675",

        "current_evidence":
            (
                "Direct/candidate Nielsen labels "
                "are available."
            ),

        "reason_excluded":
            (
                "No row-level X compatible with "
                "the current proxy feature space."
            ),

        "future_role":
            "LABEL_REFERENCE_OR_FUTURE_X_RECOVERY"
    },

    {
        "source_dataset":
            "Dashboards",

        "current_evidence":
            (
                "Behavioral X exists, but the "
                "audited feature representation "
                "does not match the current "
                "three-feature structural space."
            ),

        "reason_excluded":
            (
                "No compatible row-level "
                "structural X materialized."
            ),

        "future_role":
            "U_IN_BEHAVIOR_SPECIFIC_EXPERIMENT"
    },

    {
        "source_dataset":
            "A11y_CUA",

        "current_evidence":
            (
                "Behavior/event-derived features "
                "exist in exploratory work."
            ),

        "reason_excluded":
            (
                "Mixed web/desktop modality and "
                "no current row-level mapping to "
                "the three structural proxy X."
            ),

        "future_role":
            "U_PENDING_FEATURE_HARMONIZATION"
    },

    {
        "source_dataset":
            "P3M",

        "current_evidence":
            "Positive-only visual Nielsen issues.",

        "reason_excluded":
            (
                "Screenshot modality does not "
                "provide current structural X."
            ),

        "future_role":
            "LABEL_REFERENCE"
    }
])

display(excluded_sources)


,source_dataset,current_evidence,reason_excluded,future_role
0,29042675,Direct/candidate Nielsen labels are available.,No row-level X compatible with the current proxy feature space.,LABEL_REFERENCE_OR_FUTURE_X_RECOVERY
1,Dashboards,"Behavioral X exists, but the audited feature representation does not match the current three-feature structural space.",No compatible row-level structural X materialized.,U_IN_BEHAVIOR_SPECIFIC_EXPERIMENT
2,A11y_CUA,Behavior/event-derived features exist in exploratory work.,Mixed web/desktop modality and no current row-level mapping to the three structural proxy X.,U_PENDING_FEATURE_HARMONIZATION
3,P3M,Positive-only visual Nielsen issues.,Screenshot modality does not provide current structural X.,LABEL_REFERENCE


## 17. Registrar cambio de dominio

In [16]:
domain_registry = pd.DataFrame([
    {
        "source_dataset":
            "Catching_UX_Flaws",

        "domain":
            "WEB_DOM",

        "ssl_role":
            "L",

        "n_rows":
            len(catching_master),

        "scientific_warning":
            (
                "Very small labeled seed; "
                "two sites only."
            )
    },

    {
        "source_dataset":
            "UICrit_RICO",

        "domain":
            "ANDROID_UI",

        "ssl_role":
            "U",

        "n_rows":
            len(rico_master),

        "scientific_warning":
            (
                "Unlabeled pool comes from a "
                "different UI platform/domain. "
                "SSL assumptions must be audited "
                "before training."
            )
    }
])

display(domain_registry)


,source_dataset,domain,ssl_role,n_rows,scientific_warning
0,Catching_UX_Flaws,WEB_DOM,L,2,Very small labeled seed; two sites only.
1,UICrit_RICO,ANDROID_UI,U,1000,Unlabeled pool comes from a different UI platform/domain. SSL assumptions must be audited before training.


## 18. Decisión de cierre del notebook 26

El dataset puede materializarse como **entrada experimental** a la auditoría SSL.

No se declara entrenable todavía porque:

- L contiene únicamente dos sitios;
- las cuatro Y observadas son positivas;
- U proviene de Android mientras L proviene de Web;
- no existe un espacio canónico exacto.

El notebook 27 debe decidir si el escenario es:

- `CONVENTIONAL_SSL_READY`;
- `PU_LEARNING_CANDIDATE`;
- `DOMAIN_ADAPTATION_REQUIRED`;
- o `NOT_TRAINABLE_WITH_CURRENT_EVIDENCE`.


In [17]:
closure_decision = pd.DataFrame([
    {
        "decision":
            "MASTER_PROXY_MATERIALIZED",

        "value":
            True,

        "detail":
            (
                f"{len(master_proxy)} total rows "
                f"over {len(FEATURES)} proxy features."
            )
    },

    {
        "decision":
            "STRICT_MASTER_MATERIALIZED",

        "value":
            False,

        "detail":
            (
                "STRICT_CANONICAL is empty."
            )
    },

    {
        "decision":
            "NEGATIVE_LABELS_CREATED",

        "value":
            False,

        "detail":
            (
                "No unknown label was converted "
                "to zero."
            )
    },

    {
        "decision":
            "RICO_RATINGS_USED_AS_NIELSEN_Y",

        "value":
            False,

        "detail":
            (
                "All four Nielsen targets are "
                "NaN for UICrit/RICO."
            )
    },

    {
        "decision":
            "READY_FOR_TRAINING",

        "value":
            False,

        "detail":
            (
                "Notebook 27 must audit class "
                "support and domain-shift assumptions."
            )
    }
])

display(closure_decision)


,decision,value,detail
0,MASTER_PROXY_MATERIALIZED,True,1002 total rows over 3 proxy features.
1,STRICT_MASTER_MATERIALIZED,False,STRICT_CANONICAL is empty.
2,NEGATIVE_LABELS_CREATED,False,No unknown label was converted to zero.
3,RICO_RATINGS_USED_AS_NIELSEN_Y,False,All four Nielsen targets are NaN for UICrit/RICO.
4,READY_FOR_TRAINING,False,Notebook 27 must audit class support and domain-shift assumptions.


## 19. Guardar artefactos

In [18]:
master_proxy.to_csv(
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset.csv",
    index=False
)

target_membership.to_csv(
    SEMISUP_INTERIM_DIR
    / "semisupervised_target_membership_long.csv",
    index=False
)

split_summary.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_target_split_summary.csv",
    index=False
)

labeled_balance.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_labeled_balance.csv",
    index=False
)

target_file_registry.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_target_file_registry.csv",
    index=False
)

excluded_sources.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_excluded_source_registry.csv",
    index=False
)

domain_registry.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_domain_registry.csv",
    index=False
)

integrity_audit.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_integrity_audit.csv",
    index=False
)

closure_decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_split_closure_decision.csv",
    index=False
)


In [19]:
expected_outputs = [
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset.csv",

    SEMISUP_INTERIM_DIR
    / "semisupervised_target_membership_long.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_target_split_summary.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_labeled_balance.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_target_file_registry.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_excluded_source_registry.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_domain_registry.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_master_integrity_audit.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_split_closure_decision.csv"
]

for short in [
    "H1",
    "H4",
    "H5",
    "H6"
]:
    expected_outputs.extend([
        TARGET_DIR
        /
        f"{short}_labeled_seed.csv",

        TARGET_DIR
        /
        f"{short}_unlabeled_pool.csv"
    ])

for path in expected_outputs:
    print(
        path.relative_to(
            PROJECT_DIR
        ),
        "✅"
        if path.exists()
        else "❌"
    )


data/semisupervised/interim/semisupervised_proxy_master_dataset.csv ✅
data/semisupervised/interim/semisupervised_target_membership_long.csv ✅
reports/tables/semisupervised/semisupervised_target_split_summary.csv ✅
reports/tables/semisupervised/semisupervised_labeled_balance.csv ✅
reports/tables/semisupervised/semisupervised_target_file_registry.csv ✅
reports/tables/semisupervised/semisupervised_excluded_source_registry.csv ✅
reports/tables/semisupervised/semisupervised_domain_registry.csv ✅
reports/tables/semisupervised/semisupervised_master_integrity_audit.csv ✅
reports/tables/semisupervised/semisupervised_split_closure_decision.csv ✅
data/semisupervised/interim/targets/H1_labeled_seed.csv ✅
data/semisupervised/interim/targets/H1_unlabeled_pool.csv ✅
data/semisupervised/interim/targets/H4_labeled_seed.csv ✅
data/semisupervised/interim/targets/H4_unlabeled_pool.csv ✅
data/semisupervised/interim/targets/H5_labeled_seed.csv ✅
data/semisupervised/interim/targets/H5_unlabeled_pool.csv ✅
da

## Punto de detención

No iniciar todavía:

- Random Forest,
- Self-Training,
- Label Propagation,
- pseudo-etiquetado,
- ni evaluación train/test.

El siguiente notebook será:

`27_semisupervised_trainability_audit.ipynb`

y debe responder, con estos archivos ya materializados:

1. ¿hay soporte de ambas clases en `L`?
2. ¿es científicamente razonable aplicar PU learning?
3. ¿qué tan severo es el cambio `WEB_DOM → ANDROID_UI`?
4. ¿existe suficiente número de grupos etiquetados para reservar evaluación?
5. ¿qué experimento puede justificarse sin inventar evidencia?

Solo después de ese diagnóstico se decidirá si tiene sentido construir `28_supervised_baseline.ipynb`.
